# WeLoveReviews — sentimiento escrito frente a 4.5 estrellas

Harbor House Café. 500 reseñas de servicio. Este notebook es el documento de análisis: objetivos, exploración, limpieza, decisión de modelo, resultados y conclusiones. No hay un informe aparte para el cliente.

La account manager quiere saber, en claro, cuántas reseñas se leen como positivas, neutrales o negativas, y si ese reparto acompaña la media de **4.5 / 5**.


## Objetivos

La pregunta de negocio es si el texto dice lo mismo que el 4.5.

- Medir la media y la mezcla de estrellas humanas (1–2 negativo, 3 neutral, 4–5 positivo).
- Puntuar las 500 reseñas con el modelo fijo de Hugging Face, cargado una sola vez.
- Comparar los porcentajes y la media predicha con el 4.5.
- Revisar a mano los desacuerdos, sobre todo los falsos negativos de un modelo entrenado en productos.


## Exploración

Primero el archivo, sin modelo. El prompt de EDA pide forma, tipos, nulos, distribución del rating, longitud del texto y una muestra.


In [1]:
from pathlib import Path
import sys

import pandas as pd

pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", 40)
pd.set_option("display.width", 240)
from IPython.display import display

ROOT = Path.cwd().resolve()
if not (ROOT / "src" / "app.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

import app

reviews = pd.read_csv(ROOT / "data" / "raw" / "reviews.csv")
print("shape", reviews.shape)
print(reviews.dtypes.to_string())
print("missing")
print(reviews.isna().sum().to_string())
reviews.head(3)


shape (500, 3)
review_id      int64
rating         int64
review_text      str
missing
review_id      0
rating         0
review_text    0


,review_id,rating,review_text
0,1,5,Visited Harbor House Café last weekend. Great spot to work or catch up with friends. Every dish was bursting with flavor. Worth every penny. Already planning my next visit.
1,2,5,Tried Harbor House Café after seeing it recommended online. The coffee was rich and perfectly brewed. Great spot to work or catch up with friends. Great value for what you get. Five stars without hesitation.
2,3,5,Tried Harbor House Café after seeing it recommended online. Great spot to work or catch up with friends. The seasonal menu never disappoints. Great value for what you get. Already planning my next visit.


No hay celdas vacías. El siguiente paso comprueba si el 4.5 anunciado es la media de este archivo.


In [2]:
rating_counts = reviews["rating"].value_counts().sort_index()
rating_share = (rating_counts / len(reviews) * 100).round(2)
rating_table = pd.DataFrame({"reviews": rating_counts, "percent": rating_share})
print("mean_rating", round(float(reviews["rating"].mean()), 4))
print("stated_average", app.STATED_AVERAGE_RATING)
rating_table


mean_rating 4.5
stated_average 4.5


,reviews,percent
rating,,
1,12,2.4
2,18,3.6
3,38,7.6
4,72,14.4
5,360,72.0


La media es 4.500. El 4.5 no es un eslogan distinto del archivo. Ahora la longitud del texto, para ver si hay reseñas vacías o demasiado largas para el modelo.


In [3]:
lengths = reviews["review_text"].astype(str).str.len()
words = reviews["review_text"].astype(str).str.split().str.len()
length_table = pd.DataFrame(
    {
        "characters": lengths.describe().round(2),
        "words": words.describe().round(2),
    }
)
length_table


,characters,words
count,500.00,500.00
mean,170.17,29.10
std,27.06,4.82
min,86.00,14.00
25%,155.00,26.00
50%,173.00,29.00
75%,189.00,32.00
max,230.00,42.00


Textos cortos (entre 14 y 42 palabras). Nada se acerca al límite de 512 tokens. Una muestra de reseñas, y después los duplicados.


In [4]:
reviews.loc[:5, ["review_id", "rating", "review_text"]]


,review_id,rating,review_text
0,1,5,Visited Harbor House Café last weekend. Great spot to work or catch up with friends. Every dish was bursting with flavor. Worth every penny. Already planning my next visit.
1,2,5,Tried Harbor House Café after seeing it recommended online. The coffee was rich and perfectly brewed. Great spot to work or catch up with friends. Great value for what you get. Five stars without hesitation.
2,3,5,Tried Harbor House Café after seeing it recommended online. Great spot to work or catch up with friends. The seasonal menu never disappoints. Great value for what you get. Already planning my next visit.
3,4,5,Visited Harbor House Café last weekend. The staff were warm and attentive. The music and decor set the perfect mood. Worth every penny. Five stars without hesitation.
4,5,5,Grabbed a quick coffee at Harbor House Café this morning. They went out of their way to accommodate my allergy. The food was absolutely delicious. Five stars without hesitation.
5,6,5,Stopped by Harbor House Café for the first time. The music and decor set the perfect mood. They went out of their way to accommodate my allergy. Will definitely be back!


Varias frases se repiten de reseña en reseña. Hay que contar textos idénticos antes de decidir si se borran.


In [5]:
duplicate_rows = reviews["review_text"].duplicated(keep=False)
duplicate_texts = reviews.loc[duplicate_rows, "review_text"].nunique()
print("rows", len(reviews))
print("duplicate_text_rows", int(duplicate_rows.sum()))
print("duplicate_text_strings", int(duplicate_texts))
print("duplicate_ids", int(reviews["review_id"].duplicated().sum()))


rows 500
duplicate_text_rows 14
duplicate_text_strings 7
duplicate_ids 0


## Insights

- El 4.5 es exacto: media 4.500. El 86.4% de los ratings ya cae en la banda positiva (4–5), el 7.6% es neutral y el 6.0% es negativo.
- El texto es corto, completo y repetitivo. Comparar sentimiento con el 4.5 es viable sin entrenar nada.
- Siete textos aparecen dos veces (14 filas) con `review_id` distinto. Son registros separados, no filas rotas.
- Un 4.5 puede convivir con quejas de servicio dentro de reseñas de 5 estrellas. Eso es lo que hay que medir con el modelo, no un motivo para limpiar el texto.


## Limpieza

Solo se limpia si hace falta. Esta comprobación mira espacios, vacíos y el rango de estrellas.


In [6]:
text = reviews["review_text"].astype(str)
print("strip_changes", int((text != text.str.strip()).sum()))
print("empty_text", int(text.str.strip().eq("").sum()))
print("rating_outside_1_5", int((~reviews["rating"].between(1, 5)).sum()))


strip_changes 0
empty_text 0
rating_outside_1_5 0


No hay nada que corregir. No se recorta, no se traduce y no se eliminan duplicados: cada `review_id` se puntúa tal como llegó. Cambiar el texto cambiaría el diagnóstico.


## Plan y modelo

El archivo ya responde la mitad de la pregunta (la media humana es 4.5 y la mezcla es muy positiva). Falta la otra mitad: qué banda asigna el texto.

Se usa el modelo pedido, no otro:

- `nlptown/bert-base-multilingual-uncased-sentiment`
- Revisión fijada `8f6f4e3a8f70be4b65d3a4a8762b6d781cda240d` (no `main`)
- Salida de 1 a 5 estrellas, mapeada así: 1–2 negativo, 3 neutral, 4–5 positivo
- Carga con `pipeline()` una sola vez, fuera del bucle
- El mismo código de producción vive en `src/app.py`

El modelo viene de reseñas de productos. Aquí el texto habla de personal, esperas y ambiente. Ese desajuste puede marcar como negativas reseñas que una persona lee como positivas. Se usa igual, y los falsos negativos se revisan debajo. No se entrena ni se hace fine-tuning.


In [7]:
print(app.MODEL_ID)
print(app.MODEL_REVISION)
for stars in range(1, 6):
    print(stars, app.sentiment_band(stars))


nlptown/bert-base-multilingual-uncased-sentiment
8f6f4e3a8f70be4b65d3a4a8762b6d781cda240d
1 Negative
2 Negative
3 Neutral
4 Positive
5 Positive


## Inferencia

La siguiente celda carga el modelo una vez y puntúa las 500 reseñas con ese objeto. `annotate_reviews` recibe el pipeline ya creado; no vuelve a descargarlo.


In [8]:
pipe = app.load_sentiment_pipeline()
annotated = app.run(
    ROOT / "data" / "raw" / "reviews.csv",
    ROOT / "data" / "processed" / "reviews_with_sentiment.csv",
    pipe=pipe,
)
print("pipeline", type(pipe).__name__)
print("rows", len(annotated))
print("missing_sentiment", int(annotated["sentiment"].isna().sum()))
print(annotated["sentiment"].value_counts().to_string())
annotated[["review_id", "rating", "predicted_stars", "sentiment", "predicted_score"]].head(5)


pipeline TextClassificationPipeline
rows 500
missing_sentiment 0
sentiment
Positive    412
Negative     47
Neutral      41


,review_id,rating,predicted_stars,sentiment,predicted_score
0,1,5,5,Positive,0.716958
1,2,5,5,Positive,0.982220
2,3,5,5,Positive,0.662323
3,4,5,5,Positive,0.989547
4,5,5,5,Positive,0.982912


## Resultados frente a 4.5

Misma regla de bandas para el rating humano y para el modelo. La fila de brecha es modelo menos rating. El 4.5 es una media, así que la comparación justa es la mezcla de bandas cuya media es 4.5, más la media de estrellas predichas.


In [9]:
comparison = app.comparison_against_rating(annotated)
print("stated_average_rating", app.STATED_AVERAGE_RATING)
display(comparison)
pd.crosstab(annotated["rating"], annotated["predicted_stars"], margins=True)


stated_average_rating 4.5


,source,positive_pct,neutral_pct,negative_pct,mean_stars
0,Human rating (bands; mean is the stated 4.5),86.4,7.6,6.0,4.500
1,nlptown text sentiment,82.4,8.2,9.4,4.398
2,Gap (model − human rating),-4.0,0.6,3.4,-0.102


predicted_stars,1,2,3,4,5,All
rating,,,,,,
1,3,9,0,0,0,12
2,7,9,2,0,0,18
3,0,3,35,0,0,38
4,1,0,3,47,21,72
5,11,4,1,9,335,360
All,22,25,41,56,356,500


## Falsos negativos

Falso negativo estricto: el modelo dice 1 o 2 estrellas y la persona puso 4 o 5. También se miran los casos de rating alto que el modelo deja en neutral.


In [10]:
false_negatives = annotated[annotated["false_negative"] == 1].copy()
bursting = "Every dish was bursting with flavor"
planning = "Already planning my next visit"
both = annotated["review_text"].str.contains(bursting, regex=False) & annotated[
    "review_text"
].str.contains(planning, regex=False)
print("false_negatives", len(false_negatives))
print("median_top_score", round(float(false_negatives["predicted_score"].median()), 3))
print(
    "fn_with_bursting",
    int(false_negatives["review_text"].str.contains(bursting, regex=False).sum()),
)
print("rows_with_both_phrases", int(both.sum()))
print("both_phrases_and_false_negative", int(both[annotated["false_negative"].eq(1)].sum()))
print("mean_predicted_stars_by_human_rating")
print(annotated.groupby("rating")["predicted_stars"].mean().round(3).to_string())
false_negatives[
    ["review_id", "rating", "predicted_stars", "predicted_score", "review_text"]
].sort_values("review_id")


false_negatives 16
median_top_score 0.358
fn_with_bursting 12
rows_with_both_phrases 11
both_phrases_and_false_negative 9
mean_predicted_stars_by_human_rating
rating
1    1.750
2    1.722
3    2.921
4    4.208
5    4.814


,review_id,rating,predicted_stars,predicted_score,review_text
13,14,4,1,0.444348,Tried Harbor House Café after seeing it recommended online. Every dish was bursting with flavor. The team clearly cares about the customers. Already planning my next visit.
14,15,5,1,0.389345,"Every dish was bursting with flavor. The place has such a cozy, welcoming vibe. Already planning my next visit."
72,73,5,1,0.267875,Stopped by Harbor House Café for the first time. Service was fast and friendly. Every dish was bursting with flavor. Worth every penny. Already planning my next visit.
86,87,5,1,0.324704,Tried Harbor House Café after seeing it recommended online. The seasonal menu never disappoints. They went out of their way to accommodate my allergy. Will definitely be back!
137,138,5,2,0.275116,"Tried Harbor House Café after seeing it recommended online. They went out of their way to accommodate my allergy. The place has such a cozy, welcoming vibe. Already planning my next visit."
199,200,5,2,0.368520,Stopped by Harbor House Café for the first time. The music and decor set the perfect mood. Every dish was bursting with flavor. Already planning my next visit.
203,204,5,1,0.349168,"Came to Harbor House Café for a birthday brunch. Every dish was bursting with flavor. The cashier was rude when i asked about the menu, but honestly the food made up for it. Already planning my next visit."
253,254,5,1,0.364092,Stopped by Harbor House Café for the first time. They went out of their way to accommodate my allergy. Great spot to work or catch up with friends. Will definitely be back!
269,270,5,1,0.409078,Stopped by Harbor House Café for the first time. Our server remembered our names by the second visit. Every dish was bursting with flavor. Great value for what you get. Will definitely be back!
321,322,5,1,0.607707,"Every dish was bursting with flavor. The staff seemed annoyed to be there, but honestly the food made up for it. Will definitely be back!"


La tabla anterior guarda solo la clase ganadora. La celda siguiente reutiliza el mismo pipeline y muestra las cinco probabilidades de cuatro reseñas: un falso negativo inseguro (15), un elogio que sí acierta (1), un falso negativo con queja de personal (322) y un elogio evidente (265).


In [11]:
check_ids = [15, 1, 322, 265]
subset = annotated.set_index("review_id").loc[check_ids]
distributions = pipe(subset["review_text"].tolist(), top_k=None, truncation=True)
order = ["1 star", "2 stars", "3 stars", "4 stars", "5 stars"]
for review_id, item in zip(check_ids, distributions):
    scores = {row["label"]: round(float(row["score"]), 3) for row in item}
    print(review_id, " ".join(f"{label}={scores[label]}" for label in order))


15 1 star=0.389 2 stars=0.285 3 stars=0.102 4 stars=0.085 5 stars=0.139
1 1 star=0.022 2 stars=0.02 3 stars=0.036 4 stars=0.204 5 stars=0.717
322 1 star=0.608 2 stars=0.274 3 stars=0.065 4 stars=0.027 5 stars=0.027
265 1 star=0.002 2 stars=0.001 3 stars=0.001 4 stars=0.003 5 stars=0.994


### Qué comparten y por qué falla el modelo

De 500 reseñas, 16 son falso negativo (3.2%). Doce de esas 16 contienen "Every dish was bursting with flavor". De las 11 reseñas que juntan esa frase con "Already planning my next visit", 9 quedan en 1–2 estrellas. La mediana del score máximo en los falsos negativos es 0.358: la etiqueta negativa sale de un argmax inseguro, no de una probabilidad alta. En el id 15, por ejemplo, "1 star" tiene 0.389 y "2 stars" 0.285; el texto solo elogia el sabor, el ambiente y las ganas de volver.

Eso encaja con el desajuste de dominio, con un matiz. El modelo se afinó en productos y aquí el idioma es de servicio, a menudo en frases hechas. Dos mecanismos distintos:

1. **Argmax plano.** Elogios de servicio poco parecidos a una ficha de producto ("bursting with flavor", "accommodate my allergy", "already planning my next visit") dejan la probabilidad repartida. La banda negativa es el máximo de una distribución floja. El id 1 usa las mismas frases de riesgo y aun así sale 5 estrellas (0.717) porque añade "Worth every penny": la frase empuja, no decide sola.
2. **Queja tratada como defecto de producto.** Solo dos falsos negativos cuentan un problema de servicio y lo perdonan: el id 204 ("cashier was rude ... but honestly the food made up for it") y el id 322 ("staff seemed annoyed ... but honestly the food made up for it", score 0.608). Ahí el modelo de productos se queda con el defecto, como si el artículo viniera roto, y no con el cierre positivo del cliente.

En agregado el modelo sí sigue al rating humano: la media predicha es 1.750, 1.722, 2.921, 4.208 y 4.814 para las estrellas 1, 2, 3, 4 y 5. La brecha global es pequeña. El fallo está concentrado en un molde de frase y en unas pocas quejas de servicio dentro de reseñas altas.


## Muestra manual

Dieciocho reseñas, leídas una a una. La celda enseña la predicción; la tabla de debajo es la lectura humana y dice si la etiqueta se sostiene.


In [12]:
sample_ids = [265, 97, 1, 19, 28, 22, 9, 16, 25, 32, 140, 192, 15, 14, 87, 204, 322, 477]
sample = annotated.set_index("review_id").loc[sample_ids].reset_index()
print("sample_size", len(sample))
sample[
    ["review_id", "rating", "predicted_stars", "predicted_score", "sentiment", "review_text"]
]


sample_size 18


,review_id,rating,predicted_stars,predicted_score,sentiment,review_text
0,265,5,5,0.993982,Positive,Best brunch i've had in months. The team clearly cares about the customers. Five stars without hesitation.
1,97,5,5,0.993839,Positive,Great spot to work or catch up with friends. The pastries were incredible. Five stars without hesitation.
2,1,5,5,0.716958,Positive,Visited Harbor House Café last weekend. Great spot to work or catch up with friends. Every dish was bursting with flavor. Worth every penny. Already planning my next visit.
3,19,4,4,0.715694,Positive,"Regular customer at Harbor House Café here. The pastries were incredible. The staff were polite but a bit slow, but overall a solid experience. Might give it another shot."
4,28,5,5,0.787169,Positive,"Visited Harbor House Café last weekend. The food was absolutely delicious. Our server got our order wrong twice, but honestly the food made up for it. Highly recommend to anyone in the area."
5,22,5,5,0.421191,Positive,"Fresh ingredients you can really taste. We waited 25 minutes just to get water, but honestly the food made up for it. Will definitely be back!"
6,9,3,3,0.646989,Neutral,"Regular customer at Harbor House Café here. Average sandwiches, nothing to write home about. The staff were polite but a bit slow. It's fine for a quick stop."
7,16,3,3,0.505683,Neutral,Stopped by Harbor House Café for the first time. The menu is fine but a bit repetitive. The cashier was rude when i asked about the menu. Might give it another shot.
8,25,3,3,0.725456,Neutral,"Visited Harbor House Café last weekend. The coffee was okay. Service was fine, nothing memorable. Might give it another shot."
9,32,2,1,0.596492,Negative,Stopped by Harbor House Café for the first time. My eggs came out cold. The cashier was rude when i asked about the menu. Not sure I'll be returning.


| ID | Humano | Modelo | Lectura |
| --- | --- | --- | --- |
| 265 | 5 | 5 (0.994) | "Best brunch" y "Five stars without hesitation". La etiqueta positiva es correcta. |
| 97 | 5 | 5 (0.994) | Elogio corto y cierre explícito de cinco estrellas. Correcta. |
| 1 | 5 | 5 (0.717) | Incluye "bursting with flavor" y "Already planning my next visit", pero también "Worth every penny". Se lee positivo y el modelo acierta. |
| 19 | 4 | 4 (0.716) | Bollería muy buena, personal lento, "solid experience". Cuatro estrellas encaja con el texto. |
| 28 | 5 | 5 (0.787) | El pedido salió mal dos veces y el cliente lo perdona por la comida. El cierre es positivo. La etiqueta positiva es razonable. |
| 22 | 5 | 5 (0.421) | Esperó 25 minutos por agua y aun así volvería. La banda positiva se sostiene; la confianza es baja. |
| 9 | 3 | 3 (0.647) | Sándwiches del montón, personal correcto pero lento. Neutral es lo que dice el texto. |
| 16 | 3 | 3 (0.506) | Menú repetitivo y un cajero borde, con ganas de dar otra oportunidad. Neutral es justo. |
| 25 | 3 | 3 (0.725) | Café correcto, servicio olvidable. Correcta. |
| 32 | 2 | 1 (0.596) | Huevos fríos, cajero borde, no piensa volver. Negativo es correcto; 1 frente a 2 no cambia la banda. |
| 140 | 1 | 1 (0.507) | Café quemado, espera larga, local caótico. Negativo correcto. |
| 192 | 2 | 3 (0.658) | "Decent, nothing special" y sin ganas de volver. El texto es tibio, no un ataque. El neutral del modelo se acerca más al párrafo que el 2 del cliente. La etiqueta no está mal. |
| 15 | 5 | 1 (0.389) | Solo elogios: sabor, ambiente acogedor, piensa volver. No hay queja. Falso negativo. La etiqueta no se sostiene. |
| 14 | 4 | 1 (0.444) | Lo recomienda, elogia el sabor y al equipo, y piensa volver. Falso negativo. La etiqueta no se sostiene. |
| 87 | 5 | 1 (0.325) | Menú de temporada, le resolvieron una alergia y va a volver. Se lee positivo. Falso negativo con score plano. |
| 204 | 5 | 1 (0.349) | Hay un cajero borde, y el cliente dice que la comida lo compensó y que volverá. El conjunto sigue siendo positivo. El modelo se queda con la queja. Etiqueta demasiado dura. |
| 322 | 5 | 1 (0.608) | El personal parecía molesto, pero la comida compensa y va a volver. Es el falso negativo más seguro. El texto resuelve a favor del local; la etiqueta negativa no se sostiene. |
| 477 | 5 | 3 (0.271) | Buen café, 25 minutos de espera, la comida lo compensa y piensa volver. Se lee positivo. No es un falso negativo estricto (no cae en 1–2), pero la banda neutral tampoco describe el texto. Score casi plano. |

Doce de las dieciocho etiquetas encajan con la lectura. Las que no encajan son los ids 15, 14, 87, 204, 322 y, en menor grado, el 477. El 192 es un desacuerdo con el rating, no con el texto.


## Conclusiones

Tabla comparativa para la account manager. Las bandas usan la misma regla en las dos filas (1–2 negativo, 3 neutral, 4–5 positivo). La media humana de este archivo es el 4.5 anunciado.

| Fuente | Positivo (4–5) | Neutral (3) | Negativo (1–2) | Media de estrellas |
| --- | --- | --- | --- | --- |
| Rating humano | 86.4% | 7.6% | 6.0% | 4.500 |
| Texto, modelo nlptown | 82.4% | 8.2% | 9.4% | 4.398 |
| Brecha (modelo − rating) | −4.0 pp | +0.6 pp | +3.4 pp | −0.102 |

Se puede decir, en claro: el texto acompaña el 4.5. La media predicha es 4.398 y el 82.4% de las reseñas se lee como positivo, a cuatro puntos del 86.4% humano. No hay un relato oculto de un local mal valorado.

La brecha que sí hay que explicar antes de poner el número delante del cliente son 16 reseñas (3.2%) con rating 4–5 y predicción 1–2. Casi todas son elogios con un molde de frase ("bursting with flavor" junto a una promesa de volver) que el modelo de productos clasifica mal y con poca confianza. Dos de ellas sí traen una queja de personal que el cliente perdona. No conviene listar el 9.4% "negativo" del modelo como detractores reales sin leer esas 16 filas: `false_negative` en `data/processed/reviews_with_sentiment.csv`.

La celda siguiente vuelve a calcular la tabla desde las predicciones, para que el número escrito arriba y el número ejecutado sean el mismo.


In [13]:
comparison = app.comparison_against_rating(annotated)
print("rows", len(annotated))
print("false_negatives", int(annotated["false_negative"].sum()))
print("human_mean", round(float(annotated["rating"].mean()), 4))
print("model_mean", round(float(annotated["predicted_stars"].mean()), 4))
comparison


rows 500
false_negatives 16
human_mean 4.5
model_mean 4.398


,source,positive_pct,neutral_pct,negative_pct,mean_stars
0,Human rating (bands; mean is the stated 4.5),86.4,7.6,6.0,4.500
1,nlptown text sentiment,82.4,8.2,9.4,4.398
2,Gap (model − human rating),-4.0,0.6,3.4,-0.102
